# 第13章：软件流水线 -- 隐藏内存延迟

## 前置知识
- 第09章：分块矩阵乘法基础
- 第12章：Block Pointer 与 Shared Memory

## 学习目标
- 理解 **软件流水线 (Software Pipelining)** 的核心思想
- 理解 CUDA 中双缓冲 / 三缓冲的设计 (`simt_pipline.cu`, `mma_pipline.cu`)
- 掌握 Triton 中 `num_stages` 参数的作用
- 通过 **PTX 分析** 验证编译器生成了 `cp.async` 异步拷贝指令
- 实现不同 pipeline 深度的 GEMM 并对比性能
- 与 **cuBLAS** 进行性能对比

## 对应 CUDA 代码
- `src/simt/04simt_pipline.cu` — SIMT 双缓冲流水线
- `src/mma/04mma_pipline.cu` — MMA 双缓冲流水线
- 核心思想：加载下一个 tile 的同时计算当前 tile

## 累进优化
- ✅ Shared Memory / Block Pointer (Ch.12)
- ✅ **软件流水线 (本章新增)**

In [1]:
import torch
import triton
import triton.language as tl
import re, collections

## 13.1 内存延迟问题

### GPU 内存访问延迟

```
访问类型           延迟 (cycles)   带宽
────────────────────────────────────────────
Register           ~1              —
Shared Memory      ~20-30          ~TB/s
L2 Cache           ~200-300        ~2-4 TB/s
Global Memory      ~400-600        ~1-2 TB/s
```

### 无流水线的 GEMM (停顿问题)

```
时间线 (无流水线, num_stages=1):

K=0:    |███ Load A0,B0 ███|██ Compute ██|
K=BK:   |                  |             |███ Load A1,B1 ███|██ Compute ██|
K=2BK:  |                  |             |                  |             |███ Load ███|██ Compute ██|

        ▼ 计算单元空闲 ▼   ▼ 内存空闲 ▼  ▼ 计算单元空闲 ▼   ▼ 内存空闲 ▼

问题: 加载和计算串行执行 → 总时间 = 加载时间 + 计算时间
```

### 流水线的解决方案

```
时间线 (双缓冲流水线, num_stages=2):

加载:   |███ Load A0,B0 ███|███ Load A1,B1 ███|███ Load A2,B2 ███|
计算:   |                  |██ Compute(A0,B0)██|██ Compute(A1,B1)██|█ Comp(A2,B2)█|

        ↑ 预加载第一个 tile  ↑ 加载和计算重叠!

关键: 当计算 tile k 时, 同时加载 tile k+1
  → 总时间 ≈ max(加载时间, 计算时间) (理想情况)
```

## 13.2 CUDA Pipeline 的实现回顾

### simt_pipline.cu 的流水线结构

```
 ←-----------------------------------------------------------------------------------
 ⤷---------------------------------------iter k-----------------------------------→-⤴
 |████████████████load global███████████████████████|███store shared███|             |
 |---------------------------------------iter bk-----------------------↘-------------|
 |█load shared█|█load shared█|█load shared█|█load shared█|█load shared█|█load shared█|
 ↘-------------↘------------↘-------------↘-------------↘-------------↘-------------↘
 |████Math█████|████Math█████|████Math█████|████Math█████|████Math█████|████Math█████|
```

### CUDA 双缓冲 (Double Buffering) 的代码复杂度

```cpp
// CUDA: 需要手动管理双缓冲 (~220 行)
__shared__ float4 smem_A[2][...];   // 两个 buffer
__shared__ float4 smem_B[2][...];
float4 reg_a[2][...];               // 两个寄存器组
float4 reg_b[2][...];

// Prologue: 预加载第一个 tile
LOAD_GLOBAL(0);
STORE_SHARED(0);
__syncthreads();
LOAD_SHARED(0, 0, 0);

// Main loop: 手动交替 read/write buffer
for (int k = 1; k < K/BK; k++) {
    smem_write_idx = !smem_write_idx;  // 切换 buffer
    LOAD_GLOBAL(k);
    // ... 复杂的 prologue/epilogue 处理
}
```

### Triton 的等效代码

```python
# Triton: 整个流水线只需改一个参数!
for k in range(0, K, BLOCK_K):
    a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
    b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
    acc = tl.dot(a_tile, b_tile, acc=acc)
    a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
    b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
# + num_stages=3 在 launch 时指定
```

| 方面 | CUDA (simt_pipline) | Triton |
|------|--------------------|---------|
| 代码行数 | ~220 行 | ~10 行 |
| 缓冲管理 | 手动 smem[2], reg[2] | 编译器自动 |
| 索引切换 | idx = !idx | 编译器自动 |
| Prologue/Epilogue | 手动特殊处理 | 编译器自动 |
| 异步拷贝 | 手动 cp.async | 编译器自动 |
| 流水线深度 | 硬编码 (双缓冲) | 运行时可调 |

## 13.3 Triton 的流水线: num_stages 参数

### num_stages 的含义

```
num_stages = 1: 无流水线
  加载 → 计算 → 加载 → 计算 → ...
  SMEM 使用: 1 × (BM×BK + BK×BN) × sizeof(half)

num_stages = 2: 双缓冲 (对应 CUDA 的 smem_A[2])
  加载 tile 1 → 同时 {加载 tile 2, 计算 tile 1} → ...
  SMEM 使用: 2 × (BM×BK + BK×BN) × sizeof(half)

num_stages = 3: 三缓冲
  预加载 tile 1,2 → 同时 {加载 tile 3, 计算 tile 1} → ...
  SMEM 使用: 3 × (BM×BK + BK×BN) × sizeof(half)

num_stages = 4: 四缓冲
  SMEM 使用: 4 × (BM×BK + BK×BN) × sizeof(half)
```

### 编译器生成的指令

```
num_stages > 1 时, 编译器会在 Ampere+ GPU 上生成:

1. cp.async.ca.shared.global  (异步全局→共享内存拷贝)
   - 不阻塞计算管线
   - 硬件 DMA 引擎执行数据搬运

2. cp.async.commit_group      (提交一组异步拷贝)

3. cp.async.wait_group N      (等待第 N 组之前的拷贝完成)
   - N=0: 等待所有拷贝完成
   - N=1: 允许最后1组拷贝还在进行中
```

### Shared Memory 使用量

```
以 BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, FP16 为例:

每个 stage 需要:
  A tile: 128 × 32 × 2 bytes = 8 KB
  B tile: 32 × 128 × 2 bytes = 8 KB
  每 stage: 16 KB

num_stages=1: 16 KB
num_stages=2: 32 KB
num_stages=3: 48 KB
num_stages=4: 64 KB  (接近 SM 的 smem 上限)
num_stages=5: 80 KB  (可能超出某些 GPU 的限制!)
```

## 13.4 实现: Pipeline GEMM Kernel

在 Ch.12 的 Block Pointer kernel 基础上，**只改变一个参数** — `num_stages`。

Kernel 代码与 Ch.12 完全相同，流水线由编译器根据 `num_stages` 自动插入。

In [2]:
@triton.jit
def matmul_pipeline_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    Pipeline GEMM kernel — 在 Ch.12 Block Pointer 基础上添加流水线。

    累进优化:
      ✅ make_block_ptr (Ch.12) — 结构化内存访问
      ✅ FP32 累加器 — 数值精度
      ✅ Tensor Core (tl.dot) — 自动映射
      ✅ num_stages (本章) — 编译器自动插入流水线

    对应 CUDA simt_pipline.cu / mma_pipline.cu:
    - CUDA 需要 ~220 行手动管理双缓冲
    - Triton 只需在 launch 时指定 num_stages
    """
    # 2D grid (与 Ch.12 相同, 此时还没有 swizzle)
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # Block Pointer (Ch.12)
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0),
        block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N),
        block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )

    # FP32 累加器
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # K 循环 — 编译器根据 num_stages 自动生成流水线代码
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))

    # 写回
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))

In [3]:
def matmul_pipeline(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, num_stages=2):
    """Pipeline GEMM 的 host 端包装函数。"""
    assert a.dtype == torch.float16 and b.dtype == torch.float16
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    matmul_pipeline_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        num_stages=num_stages,
    )
    return c

In [4]:
# ========== 正确性验证 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
c_ref = torch.matmul(a, b)

print("不同 num_stages 的正确性验证:")
for stages in [1, 2, 3, 4]:
    c_tri = matmul_pipeline(a, b, num_stages=stages)
    max_err = (c_tri - c_ref).abs().max().item()
    rel_err = torch.norm(c_tri.float() - c_ref.float()) / torch.norm(c_ref.float())
    print(f"  num_stages={stages}: max_err={max_err:.4f}, rel_err={rel_err:.6f}, "
          f"correct={torch.allclose(c_tri, c_ref, atol=1.0)}")

不同 num_stages 的正确性验证:


  num_stages=1: max_err=0.0000, rel_err=0.000000, correct=True
  num_stages=2: max_err=0.0000, rel_err=0.000000, correct=True
  num_stages=3: max_err=0.0000, rel_err=0.000000, correct=True
  num_stages=4: max_err=0.0000, rel_err=0.000000, correct=True


## 13.5 PTX 分析: 验证 cp.async 指令

这是本章最重要的验证: 通过对比 `num_stages=1` 和 `num_stages=3` 的 **实际 PTX 代码**,
观察编译器如何将同步数据路径替换为异步 DMA。

### 数据搬运路径对比

```
num_stages=1 (同步路径, Ch.12):
  ld.global.v4.b32  HBM → register   ← 线程执行, 占用计算管线
  bar.sync           等待全部完成
  st.shared.v4.b32  register → SMEM   ← 线程执行
  bar.sync           等待全部完成
  ldmatrix.sync      SMEM → MMA reg
  mma.sync           Tensor Core 计算

num_stages=3 (异步路径, 本章):
  cp.async.cg.shared.global  HBM → SMEM (DMA, 绕过寄存器!)
  cp.async.commit_group       提交本批异步拷贝
  ... (编译器在此插入计算指令, 与 DMA 重叠)
  cp.async.wait_group N       等待第 N 批完成
  ldmatrix.sync                SMEM → MMA reg
  mma.sync                     Tensor Core 计算
```

关键区别:
- `cp.async.cg.shared.global` 是 **硬件 DMA**，不经过寄存器文件, SM 的计算单元完全空闲
- `.cg` = cache global = 通过 L2 cache 缓存全局内存访问
- `commit_group` 将一批 cp.async 打包为一个 "group", 后续可按 group 等待
- `wait_group N` = 等待除最后 N 个 group 外的所有拷贝完成 (N=0 全等, N=1 允许最新一批还在飞)

In [5]:
# ========== PTX 分析: num_stages=1 vs num_stages=3 实际代码 ==========
import collections

def extract_ptx(kernel_fn, *args, **kwargs):
    """从 Triton kernel 编译结果中提取 PTX。"""
    compiled = kernel_fn.warmup(*args, **kwargs)
    ptx = compiled.asm.get('ptx', '')
    if not ptx:
        for key in compiled.asm:
            if 'ptx' in key.lower():
                ptx = compiled.asm[key]
                break
    return ptx

# 提取两种配置的 PTX
ptx_args = (
    torch.float16, torch.float16, torch.float16,
    2048, 2048, 1024,
    1024, 1, 2048, 1, 2048, 1,
)
ptx_kwargs_base = dict(BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, num_warps=4)

ptx_s1 = extract_ptx(matmul_pipeline_kernel, *ptx_args,
                      **ptx_kwargs_base, num_stages=1, grid=(16, 16))
ptx_s3 = extract_ptx(matmul_pipeline_kernel, *ptx_args,
                      **ptx_kwargs_base, num_stages=3, grid=(16, 16))

lines_s1 = ptx_s1.strip().split('\n')
lines_s3 = ptx_s3.strip().split('\n')

# ── 指令计数对比 ──
def count_patterns(ptx):
    pats = {
        'cp.async':  r'cp\.async\.\w+',
        'ld.global': r'ld\.global\.\w+',
        'st.shared': r'st\.shared\.\w+',
        'ldmatrix':  r'ldmatrix\.sync\.\w+',
        'mma.sync':  r'mma\.sync\.aligned\.\w+',
        'bar.sync':  r'bar\.\w+',
    }
    return {k: len(re.findall(v, ptx)) for k, v in pats.items()}

c1, c3 = count_patterns(ptx_s1), count_patterns(ptx_s3)

print(f"{'指令':>12} | {'stages=1':>10} | {'stages=3':>10} | 变化")
print("-" * 55)
for key in ['cp.async', 'ld.global', 'st.shared', 'ldmatrix', 'mma.sync', 'bar.sync']:
    v1, v3 = c1.get(key, 0), c3.get(key, 0)
    diff = v3 - v1
    sign = '+' if diff > 0 else ''
    print(f"  {key:>12} | {v1:>10} | {v3:>10} | {sign}{diff}")
print(f"  {'PTX 行数':>12} | {len(lines_s1):>10} | {len(lines_s3):>10} | +{len(lines_s3)-len(lines_s1)}")

# ══════════════════════════════════════════════════════════════
# stages=1: 同步数据路径 (ld.global → bar.sync → st.shared)
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("  stages=1: 同步数据搬运路径")
print("="*70)
print("""
  完整路径: HBM → register → SMEM (两步, 需要 bar.sync 同步)
  每个 K 步需要加载 A tile (128×32×FP16) + B tile (32×128×FP16)
  = 8 次 ld.global.v4 (A) + 8 次 ld.global.v4 (B) → 但编译器可能合并
""")
count = 0
for i, line in enumerate(lines_s1):
    s = line.strip()
    if any(x in s for x in ['ld.global', 'st.shared', 'bar.sync']) and count < 16:
        print(f'  L{i:>4}: {s}')
        count += 1
        if 'bar.sync' in s:
            print()

# ══════════════════════════════════════════════════════════════
# stages=3: 异步数据路径 (cp.async)
# ══════════════════════════════════════════════════════════════
print(f"{'='*70}")
print("  stages=3: 异步 DMA 搬运路径 (cp.async)")
print("="*70)
print("""
  cp.async.cg.shared.global [smem_addr], [global_addr], 0x10, predicate
    - [smem_addr]: 目标 Shared Memory 地址
    - [global_addr]: 源 Global Memory 地址
    - 0x10 = 16 bytes (= 4×u32 = 128 bits, 与 ld.global.v4 等效带宽)
    - predicate: 条件执行 (用于边界检查, 避免越界访问)
    - 关键: 整条指令不经过寄存器! 硬件 DMA 直接 HBM→SMEM
""")
count = 0
for i, line in enumerate(lines_s3):
    s = line.strip()
    if 'cp.async' in s and count < 10:
        print(f'  L{i:>4}: {s}')
        count += 1
        if 'commit_group' in s:
            print(f'         ↑ commit_group: 将之前的 cp.async 打包为一个 group')
            print()

# ══════════════════════════════════════════════════════════════
# 两者共有: ldmatrix + mma.sync (SMEM → Tensor Core)
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("  两者共有: SMEM → Tensor Core (ldmatrix + mma.sync)")
print("="*70)
print("""
  无论同步还是异步搬运, 数据到达 SMEM 后的路径完全相同:
  ldmatrix.sync → mma.sync (与 Ch.12 的分析一致)
""")

print("  stages=3 中的 ldmatrix + mma.sync:")
count = 0
for i, line in enumerate(lines_s3):
    s = line.strip()
    if ('ldmatrix' in s or 'mma.sync' in s) and count < 6:
        print(f'  L{i:>4}: {s}')
        count += 1

print(f"""
{'='*70}
  关键发现
{'='*70}
  ✓ stages=1: ld.global({c1['ld.global']}) + st.shared({c1['st.shared']}) + bar.sync({c1['bar.sync']}) — 同步两步搬运
  ✓ stages=3: cp.async({c3['cp.async']}) + bar.sync({c3['bar.sync']}) — 异步 DMA, 无 ld.global/st.shared
  ✓ 两者的 mma.sync 数量相同 ({c1['mma.sync']}), 计算逻辑不变
  ✓ stages=3 的 PTX 更长 (+{len(lines_s3)-len(lines_s1)} 行): 额外的流水线 prologue/epilogue 代码
  ✓ cp.async 绕过寄存器文件 → SM 的计算单元可以同时执行 mma.sync
""")

          指令 |   stages=1 |   stages=3 | 变化
-------------------------------------------------------
      cp.async |          0 |         32 | +32
     ld.global |          8 |          0 | -8
     st.shared |          8 |          0 | -8
      ldmatrix |         16 |         16 | 0
      mma.sync |         64 |         64 | 0
      bar.sync |         18 |         19 | +1
        PTX 行数 |       1106 |       1253 | +147

  stages=1: 同步数据搬运路径

  完整路径: HBM → register → SMEM (两步, 需要 bar.sync 同步)
  每个 K 步需要加载 A tile (128×32×FP16) + B tile (32×128×FP16)
  = 8 次 ld.global.v4 (A) + 8 次 ld.global.v4 (B) → 但编译器可能合并

  L 338: @%p11 ld.global.v4.b32 { %r71, %r72, %r73, %r74 }, [ %rd154 + 0 ];
  L 345: @%p12 ld.global.v4.b32 { %r75, %r76, %r77, %r78 }, [ %rd155 + 0 ];
  L 352: @%p13 ld.global.v4.b32 { %r79, %r80, %r81, %r82 }, [ %rd156 + 0 ];
  L 359: @%p14 ld.global.v4.b32 { %r83, %r84, %r85, %r86 }, [ %rd157 + 0 ];
  L 361: bar.sync 	0;

  L 362: st.shared.v4.b32 	[%r7], {%r71, %r72, %r73, %r74};

## 13.6 性能对比: 不同 num_stages

接下来对比不同 `num_stages` 的性能, 并与 **cuBLAS** 基准进行比较。

In [6]:
# ========== benchmark 工具函数 ==========
def benchmark_vs_cublas(triton_fn, M, N, K, num_warmup=25, num_rep=100, **kwargs):
    """统一的 Triton vs cuBLAS (torch.matmul) benchmark。"""
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    flops = 2.0 * M * N * K

    # Triton
    for _ in range(num_warmup):
        triton_fn(a, b, **kwargs)
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(num_rep):
        triton_fn(a, b, **kwargs)
    e.record()
    torch.cuda.synchronize()
    ms_tri = s.elapsed_time(e) / num_rep

    # cuBLAS
    for _ in range(num_warmup):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(num_rep):
        torch.matmul(a, b)
    e.record()
    torch.cuda.synchronize()
    ms_cu = s.elapsed_time(e) / num_rep

    tflops_tri = flops / (ms_tri * 1e-3) / 1e12
    tflops_cu = flops / (ms_cu * 1e-3) / 1e12
    return ms_tri, ms_cu, tflops_tri, tflops_cu

## 13.x 统一 Shape Set 数值表

为了让 Part 3 的所有优化章节可以横向比较，本表使用与 Ch.12 和 Ch.18 相同的 7 个共享矩阵形状。

本章比较 `Ch.13 pipeline`、`Ch.12 smem` 与 `cuBLAS`，主指标是 `latency_ms`，并附带 `TFLOPS`、相对 `cuBLAS` 的速度比，以及相对前一章 `Ch.12 smem` 的速度比。

In [7]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd() / "03_matmul_optimization"):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.append(candidate_str)

from benchmark_utils import (
    BENCHMARK_SHAPES,
    add_relative_columns,
    benchmark_method,
    format_results,
    make_fp16_inputs,
)

print("Ch.13 shared-shape benchmark: pipeline vs smem vs cuBLAS")
chapter13_results = []
previous_map = {"Ch.13 pipeline": "Ch.12 smem"}

for shape in BENCHMARK_SHAPES:
    a, b = make_fp16_inputs(shape.M, shape.N, shape.K)
    c_ref = torch.matmul(a, b)
    methods = {
        "Ch.12 smem": lambda x, y: matmul_pipeline(x, y, num_stages=1),
        "Ch.13 pipeline": lambda x, y: matmul_pipeline(x, y, num_stages=3),
        "cuBLAS": lambda x, y: torch.matmul(x, y),
    }
    for method_name, fn in methods.items():
        chapter13_results.append(
            benchmark_method(method_name, fn, shape, a, b, c_ref=c_ref)
        )
    del a, b, c_ref
    torch.cuda.empty_cache()

chapter13_df = add_relative_columns(
    format_results(chapter13_results),
    previous_method_by_method=previous_map,
)
chapter13_df[[
    "shape_name",
    "category",
    "method",
    "latency_ms",
    "tflops",
    "speedup_vs_cublas",
    "speedup_vs_previous",
    "max_err",
    "passed",
]]

Ch.13 shared-shape benchmark: pipeline vs smem vs cuBLAS


,shape_name,category,method,latency_ms,tflops,speedup_vs_cublas,speedup_vs_previous,max_err,passed
0,square-2k,square,Ch.12 smem,0.065920,260.616943,0.899248,NaN,0.00,True
1,square-2k,square,Ch.13 pipeline,0.062574,274.551088,0.947327,1.053466,0.00,True
2,square-2k,square,cuBLAS,0.059278,289.816688,1.000000,NaN,0.00,True
3,square-4k,square,Ch.12 smem,0.384544,357.407622,1.102130,NaN,0.00,True
4,square-4k,square,Ch.13 pipeline,0.382920,358.923411,1.106805,1.004241,0.00,True
5,square-4k,square,cuBLAS,0.423818,324.287990,1.000000,NaN,0.00,True
6,tall-8k-x-512,tall-skinny,Ch.12 smem,0.063141,272.088251,0.937080,NaN,0.00,True
7,tall-8k-x-512,tall-skinny,Ch.13 pipeline,0.063258,271.585857,0.935350,0.998154,0.00,True
8,tall-8k-x-512,tall-skinny,cuBLAS,0.059168,290.357448,1.000000,NaN,0.00,True
9,tall-16k-x-256,tall-skinny,Ch.12 smem,0.064774,265.226219,0.938371,NaN,0.00,True


In [8]:
# ========== 不同 num_stages 的性能 ==========
M, N, K = 2048, 2048, 1024
BM, BN, BK = 128, 128, 32

print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"Block: ({BM}, {BN}, {BK})")
print(f"每 stage SMEM 使用: {(BM*BK + BK*BN)*2/1024:.1f} KB")

# 先获取 cuBLAS 基准
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
_, ms_cublas, _, tflops_cublas = benchmark_vs_cublas(matmul_pipeline, M, N, K, num_stages=2)

print(f"cuBLAS 基准: {ms_cublas:.3f}ms, {tflops_cublas:.1f} TFLOPS")
print(f"\n{'num_stages':>12} | {'SMEM(KB)':>10} | {'时间(ms)':>10} | {'TFLOPS':>8} | {'vs cuBLAS':>10}")
print("-" * 65)

for stages in [1, 2, 3, 4, 5]:
    smem_kb = stages * (BM * BK + BK * BN) * 2 / 1024
    try:
        ms_t, ms_c, tf_t, tf_c = benchmark_vs_cublas(
            matmul_pipeline, M, N, K,
            BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK, num_stages=stages)
        eff = ms_c / ms_t
        print(f"{stages:>12} | {smem_kb:>10.1f} | {ms_t:>10.3f} | {tf_t:>8.1f} | {eff:>9.0%}")
    except Exception as e:
        print(f"{stages:>12} | {smem_kb:>10.1f} | {'FAIL':>10} | — | (SMEM 超出上限?)")

矩阵规模: M=2048, N=2048, K=1024
Block: (128, 128, 32)
每 stage SMEM 使用: 16.0 KB
cuBLAS 基准: 0.033ms, 261.4 TFLOPS

  num_stages |   SMEM(KB) |     时间(ms) |   TFLOPS |  vs cuBLAS
-----------------------------------------------------------------
           1 |       16.0 |      0.033 |    258.2 |      104%
           2 |       32.0 |      0.034 |    253.4 |       97%


           3 |       48.0 |      0.033 |    259.4 |       99%
           4 |       64.0 |      0.033 |    259.1 |      100%
           5 |       80.0 |      0.037 |    230.5 |       88%


In [9]:
# ========== 不同矩阵尺寸下 pipeline 的效果 ==========
print("不同矩阵尺寸下 num_stages 的影响 (vs cuBLAS)")
print(f"{'Size':>20} | {'stage=1':>12} {'stage=2':>12} {'stage=3':>12} | {'cuBLAS':>12} | {'最优stage':>10}")
print("-" * 95)

for M, N, K in [
    (1024, 1024, 1024),
    (2048, 2048, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 1024),
    (4096, 4096, 2048),
    (4096, 4096, 4096),
]:
    results = []
    ms_cu = None
    for stages in [1, 2, 3]:
        try:
            ms_t, mc, _, _ = benchmark_vs_cublas(
                matmul_pipeline, M, N, K,
                BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, num_stages=stages)
            results.append((stages, ms_t))
            ms_cu = mc
        except:
            results.append((stages, float('inf')))

    best = min(results, key=lambda x: x[1])
    size_str = f"{M}x{N}x{K}"
    ms_strs = [f"{r[1]:>11.3f}ms" if r[1] < float('inf') else f"{'FAIL':>12}" for r in results]
    cu_str = f"{ms_cu:>11.3f}ms" if ms_cu else "N/A"
    print(f"{size_str:>20} | {ms_strs[0]} {ms_strs[1]} {ms_strs[2]} | {cu_str} | {'stage='+str(best[0]):>10}")

不同矩阵尺寸下 num_stages 的影响 (vs cuBLAS)
                Size |      stage=1      stage=2      stage=3 |       cuBLAS |    最优stage
-----------------------------------------------------------------------------------------------
      1024x1024x1024 |       0.029ms       0.023ms       0.023ms |       0.012ms |    stage=3
      2048x2048x1024 |       0.033ms       0.034ms       0.033ms |       0.033ms |    stage=1
      2048x2048x2048 |       0.064ms       0.064ms       0.062ms |       0.059ms |    stage=3


      4096x4096x1024 |       0.105ms       0.105ms       0.107ms |       0.100ms |    stage=2


      4096x4096x2048 |       0.218ms       0.208ms       0.205ms |       0.237ms |    stage=3


      4096x4096x4096 |       0.420ms       0.419ms       0.408ms |       0.458ms |    stage=3


## 13.7 BLOCK_K × num_stages 的交互效应

更大的 `BLOCK_K` 可以提高算术强度 (减少 K 循环次数), 但也增加每个 stage 的 SMEM 使用量。
两个参数需要联合调优。

In [10]:
# ========== BLOCK_K × num_stages 交叉实验 ==========
M, N, K = 2048, 2048, 2048

print(f"BLOCK_K × num_stages 交互 (M={M}, N={N}, K={K})")
print(f"{'BLOCK_K':>8} {'stages':>8} | {'SMEM(KB)':>10} {'时间(ms)':>10} {'TFLOPS':>8} | {'vs cuBLAS':>10}")
print("-" * 70)

# cuBLAS baseline
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
for _ in range(25):
    torch.matmul(a, b)
torch.cuda.synchronize()
s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
s.record()
for _ in range(100):
    torch.matmul(a, b)
e.record()
torch.cuda.synchronize()
ms_cublas = s.elapsed_time(e) / 100
flops = 2.0 * M * N * K
tf_cublas = flops / (ms_cublas * 1e-3) / 1e12
print(f"{'cuBLAS':>17} | {'—':>10} {ms_cublas:>10.3f} {tf_cublas:>8.1f} | {'baseline':>10}")
print("-" * 70)

best_ms = float('inf')
best_config = None

for bk in [16, 32, 64]:
    for stages in [1, 2, 3, 4]:
        smem_kb = stages * (128 * bk + bk * 128) * 2 / 1024
        try:
            ms_t, _, tf_t, _ = benchmark_vs_cublas(
                matmul_pipeline, M, N, K,
                BLOCK_M=128, BLOCK_N=128, BLOCK_K=bk, num_stages=stages)
            eff = ms_cublas / ms_t
            marker = " ◄ best" if ms_t < best_ms else ""
            if ms_t < best_ms:
                best_ms = ms_t
                best_config = (bk, stages)
            print(f"{bk:>8} {stages:>8} | {smem_kb:>10.1f} {ms_t:>10.3f} {tf_t:>8.1f} | {eff:>9.0%}{marker}")
        except Exception as e:
            print(f"{bk:>8} {stages:>8} | {smem_kb:>10.1f} {'FAIL':>10} {'—':>8} | (SMEM 超出上限)")

if best_config:
    print(f"\n最优配置: BLOCK_K={best_config[0]}, num_stages={best_config[1]} → {best_ms:.3f}ms")

BLOCK_K × num_stages 交互 (M=2048, N=2048, K=2048)
 BLOCK_K   stages |   SMEM(KB)     时间(ms)   TFLOPS |  vs cuBLAS
----------------------------------------------------------------------
           cuBLAS |          —      0.063    272.9 |   baseline
----------------------------------------------------------------------
      16        1 |        8.0      0.079    216.5 |       79% ◄ best
      16        2 |       16.0      0.065    263.3 |       96% ◄ best
      16        3 |       24.0      0.067    257.4 |       94%
      16        4 |       32.0      0.062    275.8 |      101% ◄ best
      32        1 |       16.0      0.062    278.8 |      102% ◄ best
      32        2 |       32.0      0.064    269.1 |       99%
      32        3 |       48.0      0.062    278.3 |      102%
      32        4 |       64.0      0.063    274.0 |      100%
      64        1 |       32.0      0.071    241.0 |       88%
      64        2 |       64.0      0.065    266.1 |       97%


      64        3 |       96.0      0.074    231.6 |       85%


      64        4 |      128.0      0.070    244.5 |       90%

最优配置: BLOCK_K=32, num_stages=1 → 0.062ms


## 13.8 总结

### 本章要点

1. **软件流水线的核心思想**：当计算当前 tile 时，同时加载下一个 tile，重叠延迟

2. **CUDA vs Triton**：
   - CUDA 需要手动实现双缓冲 (smem[2], reg[2])，手动管理索引切换，手动处理 prologue/epilogue
   - Triton 只需指定 `num_stages` 参数，编译器自动完成所有工作

3. **PTX 验证**：
   - `num_stages=1`: 使用 `ld.global` 同步加载
   - `num_stages≥2`: 使用 `cp.async.cg.shared.global` 异步加载 + `commit_group` / `wait_group`
   - 通过 PTX 分析**证实**编译器确实生成了流水线指令

4. **num_stages 的选择**：
   - `1`: 无流水线，最少 SMEM
   - `2`: 双缓冲，最常用
   - `3-4`: 更深的流水线，可能更好地隐藏延迟，但需要更多 SMEM
   - 受 Shared Memory 容量限制

5. **BLOCK_K × num_stages 联合调优**：
   - 更大的 BLOCK_K 提高算术强度，但增加 SMEM 使用
   - 需要联合搜索找到最优组合

### 累进优化状态
| 特性 | 状态 |
|------|------|
| Shared Memory / Block Pointer | ✅ Ch.12 |
| **软件流水线 (num_stages)** | **✅ 本章** |
| Split-K 并行 | → Ch.14 |
| L2 Cache Swizzle | → Ch.15 |

### 练习

1. **PTX 深入**：在 PTX 中找到 `cp.async.wait_group` 指令, 分析 wait 的 group 数量与 num_stages 的关系
2. **K 维度影响**：固定 M=N=2048, 改变 K (256, 512, 1024, 2048, 4096), 观察不同 num_stages 的效果
3. **SMEM 上限实验**：增大 BLOCK_M/BLOCK_N 直到 num_stages=2 也无法编译, 观察报错信息
4. **思考题**：为什么在 K 很小时 (如 K=64), 流水线可能反而更慢? (提示: prologue 开销)

### 下一章预告

第14章将介绍 **Split-K 并行**, 通过在 K 维度引入并行性来加速 tall-skinny 矩阵乘法。
Split-K 将在本章 pipeline kernel 的基础上累进叠加。